# Data

One task: help the student, but do not do the work for them. Every row is a student's
request, the expected decision (`help` or `decline`), a good answer and a bad one.
Traps are help requests that sound like violations: shorten my own draft, check my
calculation, how do I cite properly. A model that learned "when in doubt, refuse" fails there.

The system prompt says nothing about rules, so the base model does whatever it is asked.
The boundary has to come from training.

In [ ]:
import sys
sys.path.insert(0, "..")

from collections import Counter

from src import data

rows = data.rows()
for split in ("train", "test"):
    part = [r for r in rows if r["split"] == split]
    print(f"{split:5} {len(part):3}  decline {sum(r['decision'] == 'decline' for r in part):3}"
          f"  help {sum(r['decision'] == 'help' for r in part):3}  traps {sum(r['trap'] for r in part):3}")

In [ ]:
for (decision, topic), n in sorted(Counter((r["decision"], r["topic"]) for r in rows).items()):
    print(f"{decision:8} {topic:28} {n:3}")

## Rows

In [ ]:
def show(row):
    print("=" * 78)
    print(f"{row['decision'].upper()}{' · trap' if row['trap'] else ''} · {row['topic']}")
    print("REQUEST:", row["request"])
    print("GOOD:   ", row["good"])
    print("BAD:    ", row["bad"])


show(next(r for r in rows if r["decision"] == "decline"))
show(next(r for r in rows if r["decision"] == "help" and not r["trap"]))
show(next(r for r in rows if r["trap"]))

## Traps

In [ ]:
for row in [r for r in rows if r["trap"]][:15]:
    print(" ", row["request"][:110])

## What the trainers see

`data.pairs` renders the TRL conversational preference format, `data.sft` the prompt-completion view.

In [ ]:
print(data.pairs(rows[:1])[0])
print()
print(data.sft(rows[:1])[0])

## Prompts

In [ ]:
print("neutral:", data.neutral)
print()
print("strict:", data.strict)